# 03 — MAPREDUCE DI ATAS YARN
### Model pemrograman MapReduce · manajemen sumber daya YARN · pembanding Spark

**Peta ke laporan:** Bab 2.3 *Processing Framework* · LO 3

Zona silver sudah siap di notebook 02. Sebelum pipeline beralih ke Spark untuk
seluruh tahap berikutnya, lapisan platformnya diperiksa lebih dulu di sini:
**YARN** sebagai manajer sumber daya klaster, dan **MapReduce** sebagai model
pemrograman yang dijalankan sungguhan, bukan sekadar disebut namanya.

Urutannya sengaja begini. MapReduce adalah mesin pemrosesan asli Hadoop, dan
Spark hadir sebagai penggantinya. Menjalankan yang asli lebih dulu membuat
keputusan memakai Spark pada notebook 04 sampai 07 menjadi kesimpulan yang
**diukur**, bukan asumsi yang diwarisi.

Pekerjaannya pun dipilih yang akan dipakai lagi: **pencacahan *k*-mer** —
komputasi yang persis sama dengan yang dibangun notebook 04 memakai Spark.
Dengan begitu dua mesin yang berbeda total mengerjakan soal yang sama, dan
hasilnya bisa diadu angka per angka.

---
## Kenapa bab ini ada

Ada dua hal yang sering tertukar, dan membedakannya penting saat sidang.

**Model pemrograman MapReduce** adalah gagasan: pecah pekerjaan menjadi tahap
*map* yang independen per rekaman, lalu tahap *reduce* yang menggabungkan hasil
berdasarkan kunci. Gagasan ini **memang dipakai di seluruh proyek ini** —
ekstraksi *k*-mer di notebook 04 persis berbentuk itu.

**Mesin MapReduce milik Hadoop** adalah implementasi konkret gagasan tersebut:
job Java yang menulis hasil antara ke disk di setiap tahap. Inilah yang belum
dipakai, dan yang dijalankan di sini.

Spark menggantikan mesinnya, bukan gagasannya. Notebook ini membuktikan
keduanya menghasilkan jawaban yang sama, lalu mengukur selisih ongkosnya.

### Kenapa mapper-nya ditulis dalam awk

Image `apache/hadoop:3.3.6` berbasis CentOS 7 yang sudah habis masa dukungan:
cermin repositorinya dimatikan, sehingga `yum install python3` gagal dengan
`No package python3 available`. Memasang Python berarti membangun image sendiri
atau menarik ratusan megabita ke setiap NodeManager.

`awk` sudah ada di setiap node Hadoop tanpa dipasang, dan untuk pekerjaan
menggeser jendela sepanjang untai DNA kemampuannya lebih dari cukup. Hadoop
Streaming memang dirancang untuk ini: mapper dan reducer boleh berupa program
apa pun yang membaca stdin dan menulis stdout.

## Bootstrap dan keadaan klaster

Klaster harus dinyalakan dengan profil `yarn`:

```powershell
cd D:\BDA\docker
docker compose --profile standalone down
docker compose --profile yarn up -d
```

Profil `yarn` dan `standalone` tidak bisa hidup bersamaan: RAM WSL 48 GB,
sedangkan pekerja masing-masing profil meminta sekitar 25 GB.

In [1]:
import sys
sys.path.insert(0, r"D:\BDA\nb" if sys.platform == "win32" else "/workspace/nb")
from bda_common import *

import re
import subprocess
import pandas as pd
from pyspark.sql import functions as F

info_mesin()
print()
print("KLASTER YARN")
print("=" * 62)
info_yarn()

CPU logis      : 24
RAM total      : 50.5 GB   bebas 40.4 GB
Disk D: bebas  : 201.4 GB dari 1,024.1 GB
Python         : 3.10.12
Mode           : KLASTER  (hdfs://namenode:8020)
run_id         : run_20260922T021945Z

KLASTER YARN
  ResourceManager : resourcemanager:8088
  NodeManager     : 3 aktif, 0 hilang
  Memori          : 21,504 MB total, 10,752 MB bebas
  vCore           : 12 total, 6 bebas
  Aplikasi        : 1 berjalan, 1 selesai, 0 gagal


{'appsSubmitted': 2,
 'appsCompleted': 1,
 'appsPending': 0,
 'appsRunning': 1,
 'appsFailed': 0,
 'appsKilled': 0,
 'reservedMB': 0,
 'availableMB': 10752,
 'allocatedMB': 10752,
 'pendingMB': 14848,
 'reservedVirtualCores': 0,
 'availableVirtualCores': 6,
 'allocatedVirtualCores': 6,
 'pendingVirtualCores': 9,
 'containersAllocated': 6,
 'containersReserved': 0,
 'containersPending': 9,
 'totalMB': 21504,
 'totalVirtualCores': 12,
 'utilizedMBPercent': 20,
 'utilizedVirtualCoresPercent': 25,
 'rmSchedulerBusyPercent': 0,
 'totalNodes': 3,
 'lostNodes': 0,
 'unhealthyNodes': 0,
 'decommissioningNodes': 0,
 'decommissionedNodes': 0,
 'rebootedNodes': 0,
 'activeNodes': 3,
 'shutdownNodes': 0,
 'totalUsedResourcesAcrossPartition': {'memory': 10752,
  'vCores': 6,
  'resourceInformations': {'resourceInformation': [{'attributes': {},
     'maximumAllocation': 9223372036854775807,
     'minimumAllocation': 0,
     'name': 'memory-mb',
     'resourceType': 'COUNTABLE',
     'units': 'Mi',
 

---
## Program MapReduce yang dijalankan

Dua berkas, masing-masing beberapa baris. Komentar di kepalanya menjelaskan
keputusan yang diambil — terutama soal batas baris FASTA, yang kalau diabaikan
membuang sekitar 10% *k*-mer tanpa satu pun pesan galat.

In [2]:
MR = BASE / "docker" / "mr"

for nama in ("kmer_map.awk", "kmer_reduce.awk"):
    print("=" * 70)
    print(f"  {nama}")
    print("=" * 70)
    print((MR / nama).read_text(encoding="utf-8"))

  kmer_map.awk
# ═══════════════════════════════════════════════════════════════════
# MAPPER — memecah sekuens FASTA menjadi k-mer
# ═══════════════════════════════════════════════════════════════════
#
# Sisi *map* dari pencacahan k-mer. Satu rekaman sekuens masuk, sejumlah
# pasangan `k-mer <TAB> 1` keluar. Reducer yang menjumlahkannya.
#
# Kenapa awk dan bukan Python. Image apache/hadoop:3.3.6 berbasis
# CentOS 7 yang sudah habis masa dukungannya: cermin repositorinya sudah
# dimatikan, sehingga `yum install python3` gagal dengan "No package
# python3 available". Memasang Python berarti membangun image sendiri
# atau menarik Miniforge ratusan megabita ke tiap NodeManager. awk sudah
# ada di setiap node Hadoop tanpa dipasang, dan untuk pekerjaan
# geser-jendela seperti ini kemampuannya lebih dari cukup.
#
# Jebakan yang khas FASTA. Satu rekaman terdiri dari baris deskripsi
# berawalan ">" lalu sejumlah baris basa yang dipecah rata pada lebar
# tertentu. Pemecahan itu murni kosmetik 

---
## Menjalankan job

`SKALA` menentukan berapa banyak partisi arsip yang diproses. Ongkos MapReduce
di sini tumbuh linear terhadap jumlah basa — mapper memancarkan satu baris per
*k*-mer — jadi naikkan bertahap dan perhatikan waktunya.

| skala | partisi | perkiraan sekuens |
|---|---|---|
| `kecil` | 1980-an + 1990-an | ~2.600 |
| `sedang` | ditambah 2000-an | ~126.000 |
| `besar` | seluruh arsip | ~1.627.000 |

`besar` sengaja tidak dijadikan bawaan. Kalau dijalankan, waktunya sendiri yang
menjadi temuan bab ini.

In [3]:
SKALA = "kecil"
K = CFG["k_utama"]

POLA = {
    "kecil":  [f"{HDFS_URI}/bda/lake/fasta/flua_198*.fasta.gz",
               f"{HDFS_URI}/bda/lake/fasta/flua_199*.fasta.gz"],
    "sedang": [f"{HDFS_URI}/bda/lake/fasta/flua_198*.fasta.gz",
               f"{HDFS_URI}/bda/lake/fasta/flua_199*.fasta.gz",
               f"{HDFS_URI}/bda/lake/fasta/flua_200*.fasta.gz"],
    "besar":  [f"{HDFS_URI}/bda/lake/fasta/flua_*.fasta.gz"],
}

JAR = "/opt/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.3.6.jar"
TUJUAN_MR = f"/bda/mr/kmer_k{K}_{SKALA}"


def jalankan_mapreduce(pola_input, tujuan, k=K, reduces=6, nama=None):
    '''Submit job Hadoop Streaming ke YARN; kembalikan counter dan durasi.

    Hadoop menolak menulis ke direktori yang sudah ada -- sengaja, supaya
    hasil lama tidak tertimpa diam-diam. Jadi tujuannya dibersihkan dulu
    agar sel ini aman dijalankan berulang kali.
    '''
    subprocess.run(["hdfs", "dfs", "-rm", "-r", "-skipTrash", tujuan],
                   capture_output=True, text=True)

    perintah = ["hadoop", "jar", JAR,
                "-D", f"mapreduce.job.name={nama or f'BDA-kmer-k{k}-{SKALA}'}",
                "-D", f"mapreduce.job.reduces={reduces}",
                "-files", f"{MR / 'kmer_map.awk'},{MR / 'kmer_reduce.awk'}"]
    for p in pola_input:
        perintah += ["-input", p]
    perintah += ["-output", tujuan,
                 "-mapper", f"awk -v K={k} -f kmer_map.awk",
                 # Combiner memakai reducer yang sama. Penjumlahan bersifat
                 # asosiatif, jadi menjumlahkan sebagian di sisi mapper
                 # memberi hasil identik -- sambil memangkas lalu lintas
                 # shuffle secara drastis.
                 "-combiner", "awk -f kmer_reduce.awk",
                 "-reducer", "awk -f kmer_reduce.awk"]

    t0 = time.time()
    r = subprocess.run(perintah, capture_output=True, text=True)
    detik = time.time() - t0
    log = (r.stdout or "") + (r.stderr or "")

    if r.returncode != 0:
        print(log[-3000:])
        raise RuntimeError(f"job MapReduce gagal (returncode {r.returncode})")

    counter = {}
    for baris in log.splitlines():
        m = re.match(r"^\s+([^=]+?)=(\d+)\s*$", baris)
        if m:
            counter[m.group(1).strip()] = int(m.group(2))
    app = re.search(r"(application_\d+_\d+)", log)

    return {"detik": round(detik, 1),
            "app": app.group(1) if app else None,
            "counter": counter}


print(f"  Skala   : {SKALA}")
print(f"  k       : {K}")
print(f"  Masukan : {len(POLA[SKALA])} pola")
for p in POLA[SKALA]:
    print(f"            {p}")
print(f"  Tujuan  : {TUJUAN_MR}")

  Skala   : kecil
  k       : 8
  Masukan : 2 pola
            hdfs://namenode:8020/bda/lake/fasta/flua_198*.fasta.gz
            hdfs://namenode:8020/bda/lake/fasta/flua_199*.fasta.gz
  Tujuan  : /bda/mr/kmer_k8_kecil


In [4]:
with Tahap(f"MapReduce pencacahan k-mer (skala {SKALA})", "MAPREDUCE"):
    hasil_mr = jalankan_mapreduce(POLA[SKALA], TUJUAN_MR)
    c = hasil_mr["counter"]

    print(f"\n  Aplikasi YARN   : {hasil_mr['app']}")
    print(f"  Durasi          : {hasil_mr['detik']:,.1f} detik")
    print(f"  Mapper          : {c.get('Launched map tasks', 0)}")
    print(f"  Reducer         : {c.get('Launched reduce tasks', 0)}")

    mo = c.get("Map output records", 0)
    co = c.get("Combine output records", 0)
    ro = c.get("Reduce output records", 0)
    print(f"\n  Map output      : {mo:,} k-mer dipancarkan")
    if mo and co:
        print(f"  Combine output  : {co:,}  "
              f"-> lalu lintas shuffle turun {100 * (1 - co / mo):.1f}%")
    print(f"  Reduce output   : {ro:,} k-mer unik")

    byt = c.get("Bytes Written", 0)
    if byt:
        print(f"  Ditulis ke HDFS : {byt / 1e6:,.1f} MB")


--------------------------------------------------------------------
[>] MAPREDUCE | MapReduce pencacahan k-mer (skala kecil)

  Aplikasi YARN   : application_1790042040111_0003
  Durasi          : 42.4 detik
  Mapper          : 15
  Reducer         : 6

  Map output      : 2,964,548 k-mer dipancarkan
  Combine output  : 265,159  -> lalu lintas shuffle turun 91.1%
  Reduce output   : 56,171 k-mer unik
  Ditulis ke HDFS : 0.7 MB
[<] OK | 44.5 detik | RAM bebas 40.8 GB


### Membaca combiner-nya

Selisih antara *Map output records* dan *Combine output records* adalah angka
paling penting di tabel di atas. Setiap baris yang tersisa harus diurutkan,
dikirim lewat jaringan, lalu diurutkan lagi di sisi reducer. Tahap *shuffle and
sort* itulah yang biasanya mendominasi ongkos sebuah job MapReduce — dan
combiner memangkasnya sebelum satu byte pun menyeberang.

---
## Pembanding: komputasi yang sama di Spark

Spark dijalankan **di atas YARN yang sama**, bukan penjadwal lain, supaya yang
dibandingkan benar-benar mesinnya dan bukan klasternya.

Satu catatan teknis yang menentukan: seluruh transformasi ditulis sebagai Spark
SQL asli, tanpa UDF Python. Akibatnya executor cukup menjalankan JVM, dan
NodeManager berbasis CentOS 7 yang tidak punya Python sama sekali tetap bisa
menjalankannya.

In [5]:
spark = spark_session(
    "03-mapreduce-yarn",
    master="yarn",
    konfig={
        "spark.executor.instances": "3",
        "spark.executor.memory": "3g",
        "spark.executor.cores": "2",
        "spark.yarn.am.memory": "1g",
        # Driver berada di container jupyter; executor di NodeManager harus
        # bisa menghubunginya balik lewat nama host itu.
        "spark.driver.host": os.environ.get("HOSTNAME", "jupyter"),
    })
print("  master :", spark.sparkContext.master)
print("  appId  :", spark.sparkContext.applicationId)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/22 02:20:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/22 02:20:32 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.


Spark 3.5.9 | master yarn | driver 8g | paralelisme 6
Spark UI: http://jupyter:4040
  master : yarn
  appId  : application_1790042040111_0004


In [6]:
with Tahap(f"Spark SQL pencacahan k-mer (skala {SKALA})", "SPARK"):
    t0 = time.time()

    # lineSep='>' membuat satu baris DataFrame berisi satu rekaman FASTA utuh,
    # sehingga k-mer yang melintasi batas baris tidak hilang -- masalah yang di
    # sisi MapReduce ditangani penyangga di dalam mapper.
    teks = spark.read.option("lineSep", ">").text(POLA[SKALA])

    rekaman = (teks.filter(F.length("value") > 0)
                   .withColumn("seq", F.upper(F.regexp_replace(
                       F.split(F.col("value"), "\n", 2)[1], r"\s", ""))))

    gen = (f"transform(sequence(0, length(seq) - {K}), "
           f"i -> substring(seq, i + 1, {K}))")
    kmer = (rekaman.withColumn("kmers", F.expr(gen))
                   .select(F.explode("kmers").alias("kmer"))
                   .filter(F.col("kmer").rlike("^[ACGT]+$")))

    spark_hasil = kmer.groupBy("kmer").agg(F.count("*").alias("jumlah_spark"))
    spark_hasil.write.mode("overwrite").parquet(
        jalur(f"mr/spark_kmer_k{K}_{SKALA}"))

    detik_spark = round(time.time() - t0, 1)
    n_rekaman = rekaman.count()
    n_unik_spark = spark.read.parquet(jalur(f"mr/spark_kmer_k{K}_{SKALA}")).count()

    print(f"\n  Rekaman diproses : {n_rekaman:,}")
    print(f"  k-mer unik       : {n_unik_spark:,}")
    print(f"  Durasi           : {detik_spark:,.1f} detik")


--------------------------------------------------------------------
[>] SPARK | Spark SQL pencacahan k-mer (skala kecil)



  Rekaman diproses : 2,575
  k-mer unik       : 56,171
  Durasi           : 65.7 detik
[<] OK | 66.8 detik | RAM bebas 35.6 GB


---
## Apakah keduanya menjawab hal yang sama?

Pertanyaan ini bukan formalitas. Dua implementasi yang berbeda total — awk di
atas Hadoop Streaming, dan Spark SQL di dalam JVM — hanya berguna sebagai
pembanding kalau jawabannya identik. Kalau berbeda, yang diukur bukan kecepatan
melainkan dua komputasi yang berlainan.

In [7]:
with Tahap("bandingkan hasil MapReduce dan Spark", "SPARK"):
    mr_hasil = (spark.read.option("sep", "\t")
                     .schema("kmer string, jumlah_mr long")
                     .csv(f"{HDFS_URI}{TUJUAN_MR}/part-*"))
    sp_hasil = spark.read.parquet(jalur(f"mr/spark_kmer_k{K}_{SKALA}"))

    n_mr = mr_hasil.count()
    n_sp = sp_hasil.count()

    gabung = mr_hasil.join(sp_hasil, "kmer", "full_outer")
    n_beda = gabung.filter(
        F.coalesce(F.col("jumlah_mr"), F.lit(-1))
        != F.coalesce(F.col("jumlah_spark"), F.lit(-1))).count()

    total_mr = mr_hasil.agg(F.sum("jumlah_mr")).collect()[0][0] or 0
    total_sp = sp_hasil.agg(F.sum("jumlah_spark")).collect()[0][0] or 0

    print(f"  k-mer unik MapReduce : {n_mr:,}")
    print(f"  k-mer unik Spark     : {n_sp:,}")
    print(f"  Total cacah MapReduce: {total_mr:,}")
    print(f"  Total cacah Spark    : {total_sp:,}")
    print(f"\n  Baris yang berbeda   : {n_beda:,}"
          f"   {'-> IDENTIK' if n_beda == 0 else '-> PERIKSA LAGI'}")

    if n_beda == 0:
        print("\n  Dua mesin yang sama sekali berbeda menghasilkan jawaban yang")
        print("  sama persis. Perbandingan waktu di bawah karenanya sah.")


--------------------------------------------------------------------
[>] SPARK | bandingkan hasil MapReduce dan Spark
  k-mer unik MapReduce : 56,171
  k-mer unik Spark     : 56,171
  Total cacah MapReduce: 2,964,548
  Total cacah Spark    : 2,964,548

  Baris yang berbeda   : 0   -> IDENTIK

  Dua mesin yang sama sekali berbeda menghasilkan jawaban yang
  sama persis. Perbandingan waktu di bawah karenanya sah.
[<] OK | 2.2 detik | RAM bebas 35.3 GB


In [8]:
with Tahap("tabel perbandingan", "MONITORING"):
    c = hasil_mr["counter"]
    banding = pd.DataFrame([
        {"mesin": "MapReduce (Hadoop Streaming + awk)",
         "penjadwal": "YARN",
         "detik": hasil_mr["detik"],
         "k_mer_unik": n_mr,
         "hasil_antara": "ditulis ke disk tiap tahap",
         "tugas": f"{c.get('Launched map tasks', 0)} map / "
                  f"{c.get('Launched reduce tasks', 0)} reduce"},
        {"mesin": "Spark SQL",
         "penjadwal": "YARN",
         "detik": detik_spark,
         "k_mer_unik": n_sp,
         "hasil_antara": "ditahan di memori antar tahap",
         "tugas": "3 executor x 2 core"},
    ])
    display(banding)

    urut = sorted([("MapReduce", hasil_mr["detik"]), ("Spark SQL", detik_spark)],
                  key=lambda x: x[1])
    (nama_cepat, t_cepat), (nama_lambat, t_lambat) = urut
    if t_cepat > 0:
        print(f"\n  Pada skala '{SKALA}': {nama_cepat} {t_lambat / t_cepat:.1f}x "
              f"lebih cepat ({t_cepat:,.1f} detik lawan {t_lambat:,.1f} detik).")

    print()
    print("  Cara menafsirkannya, dan ini penting supaya tidak salah simpul:")
    print()
    print("  Pada skala kecil, yang terukur sebagian besar adalah ONGKOS MEMULAI,")
    print("  bukan ongkos menghitung. Spark harus meminta ApplicationMaster ke")
    print("  YARN, menunggu executor dialokasikan, lalu menyusun rencana kueri --")
    print("  puluhan detik habis sebelum baris pertama dibaca. MapReduce memulai")
    print("  jauh lebih murah: task-nya proses JVM pendek yang langsung jalan")
    print("  begitu container siap. Jadi MapReduce memang bisa menang di sini,")
    print("  dan itu hasil yang sah, bukan anomali yang perlu disembunyikan.")
    print()
    print("  Keunggulan Spark tidak terletak pada satu tahap tunggal seperti ini,")
    print("  melainkan pada pipeline BERLAPIS di atas data yang sama. MapReduce")
    print("  menuliskan hasil antara ke disk pada setiap tahap, sehingga sepuluh")
    print("  tahap berarti sepuluh kali tulis-baca disk. Spark menahannya di")
    print("  memori dan membayar ongkos itu sekali. Notebook 04 sampai 06 persis")
    print("  berbentuk begitu, dan di sanalah selisihnya menjadi menentukan.")
    print()
    print("  Naikkan SKALA ke 'sedang' lalu 'besar' untuk melihat di titik mana")
    print("  ongkos memulai tenggelam oleh ongkos menghitung.")



--------------------------------------------------------------------
[>] MONITORING | tabel perbandingan


,mesin,penjadwal,detik,k_mer_unik,hasil_antara,tugas
0,MapReduce (Hadoop Streaming + awk),YARN,42.4,56171,ditulis ke disk tiap tahap,15 map / 6 reduce
1,Spark SQL,YARN,65.7,56171,ditahan di memori antar tahap,3 executor x 2 core



  Pada skala 'kecil': MapReduce 1.5x lebih cepat (42.4 detik lawan 65.7 detik).

  Cara menafsirkannya, dan ini penting supaya tidak salah simpul:

  Pada skala kecil, yang terukur sebagian besar adalah ONGKOS MEMULAI,
  bukan ongkos menghitung. Spark harus meminta ApplicationMaster ke
  YARN, menunggu executor dialokasikan, lalu menyusun rencana kueri --
  puluhan detik habis sebelum baris pertama dibaca. MapReduce memulai
  jauh lebih murah: task-nya proses JVM pendek yang langsung jalan
  begitu container siap. Jadi MapReduce memang bisa menang di sini,
  dan itu hasil yang sah, bukan anomali yang perlu disembunyikan.

  Keunggulan Spark tidak terletak pada satu tahap tunggal seperti ini,
  melainkan pada pipeline BERLAPIS di atas data yang sama. MapReduce
  menuliskan hasil antara ke disk pada setiap tahap, sehingga sepuluh
  tahap berarti sepuluh kali tulis-baca disk. Spark menahannya di
  memori dan membayar ongkos itu sekali. Notebook 04 sampai 06 persis
  berbentuk begitu, dan

---
## Bukti bahwa semuanya melewati YARN

Daftar di bawah dibaca langsung dari REST API ResourceManager. Kolom `jenis`
membedakan keduanya: `MAPREDUCE` untuk job Hadoop Streaming, `SPARK` untuk sesi
Spark. Kalau sebuah job tidak muncul di sini, job itu tidak pernah menyentuh
klaster — dan itulah yang terjadi kalau `mapreduce.framework.name` tertinggal
pada nilai bawaannya, `local`.

Antarmuka webnya: **http://localhost:8088**

In [9]:
with Tahap("aplikasi yang tercatat di YARN", "MONITORING"):
    display(yarn_aplikasi(batas=10))
    print()
    info_yarn()


--------------------------------------------------------------------
[>] MONITORING | aplikasi yang tercatat di YARN


,id,nama,jenis,status,hasil,detik,memori_MB_detik,vcore_detik
0,application_1790042040111_0001,BDA-Influenza-02-storage-quality,SPARK,FINISHED,SUCCEEDED,117.3,1733091,341
1,application_1790042040111_0002,BDA-kmer-k8-kecil,MAPREDUCE,FINISHED,SUCCEEDED,24.3,219878,115
2,application_1790042040111_0003,BDA-kmer-k8-kecil,MAPREDUCE,FINISHED,SUCCEEDED,38.2,257251,131
3,application_1790042040111_0004,BDA-Influenza-03-mapreduce-yarn,SPARK,RUNNING,UNDEFINED,78.1,900473,290



  ResourceManager : resourcemanager:8088
  NodeManager     : 3 aktif, 0 hilang
  Memori          : 21,504 MB total, 9,216 MB bebas
  vCore           : 12 total, 8 bebas
  Aplikasi        : 1 berjalan, 3 selesai, 0 gagal
[<] OK | 0.0 detik | RAM bebas 35.3 GB


In [10]:
with Tahap("simpan ringkasan", "MONITORING"):
    c = hasil_mr["counter"]
    ringkas = {
        "run_id": RUN_ID,
        "skala": SKALA,
        "k": K,
        "mapreduce": {
            "aplikasi": hasil_mr["app"],
            "detik": hasil_mr["detik"],
            "map_tasks": c.get("Launched map tasks"),
            "reduce_tasks": c.get("Launched reduce tasks"),
            "map_output_records": c.get("Map output records"),
            "combine_output_records": c.get("Combine output records"),
            "reduce_output_records": c.get("Reduce output records"),
        },
        "spark": {
            "aplikasi": spark.sparkContext.applicationId,
            "detik": detik_spark,
            "k_mer_unik": int(n_sp),
        },
        "hasil_identik": bool(n_beda == 0),
    }
    (BASE / "models").mkdir(parents=True, exist_ok=True)
    (BASE / "models" / "ringkasan_mapreduce.json").write_text(
        json.dumps(ringkas, indent=2), encoding="utf-8")
    print(json.dumps(ringkas, indent=2))


--------------------------------------------------------------------
[>] MONITORING | simpan ringkasan
{
  "run_id": "run_20260922T021945Z",
  "skala": "kecil",
  "k": 8,
  "mapreduce": {
    "aplikasi": "application_1790042040111_0003",
    "detik": 42.4,
    "map_tasks": 15,
    "reduce_tasks": 6,
    "map_output_records": 2964548,
    "combine_output_records": 265159,
    "reduce_output_records": 56171
  },
  "spark": {
    "aplikasi": "application_1790042040111_0004",
    "detik": 65.7,
    "k_mer_unik": 56171
  },
  "hasil_identik": true
}
[<] OK | 0.0 detik | RAM bebas 35.3 GB


In [11]:
display(jejak_df())
stop_spark()
print()
print("Selesai. Lanjut ke 04_kmer_features.ipynb")
print()
print("Notebook 04 dan seterusnya memakai Spark. Kalau ingin menjalankannya")
print("dengan profil standalone, kembalikan klasternya lebih dulu:")
print("  docker compose --profile yarn down")
print("  docker compose --profile standalone up -d")
print()
print("Spark juga bisa tetap berjalan di atas YARN -- sudah dibuktikan di sel")
print("pembanding di atas -- selama seluruh transformasi memakai Spark SQL")
print("asli tanpa UDF Python, karena NodeManager berbasis CentOS 7 tidak")
print("membawa Python sama sekali.")

,run_id,lapisan,tahap,status,detik,ram_delta_gb,ram_bebas_gb,waktu
0,run_20260922T021945Z,MAPREDUCE,MapReduce pencacahan k-mer (skala kecil),OK,44.51,-0.45,40.8,2026-09-22T02:20:30.499561+00:00
1,run_20260922T021945Z,SPARK,Spark SQL pencacahan k-mer (skala kecil),OK,66.83,4.02,35.6,2026-09-22T02:21:50.025830+00:00
2,run_20260922T021945Z,SPARK,bandingkan hasil MapReduce dan Spark,OK,2.19,0.36,35.3,2026-09-22T02:21:52.223179+00:00
3,run_20260922T021945Z,MONITORING,tabel perbandingan,OK,0.01,0.00,35.3,2026-09-22T02:21:52.242645+00:00
4,run_20260922T021945Z,MONITORING,aplikasi yang tercatat di YARN,OK,0.02,0.00,35.3,2026-09-22T02:21:52.268562+00:00
5,run_20260922T021945Z,MONITORING,simpan ringkasan,OK,0.01,-0.00,35.3,2026-09-22T02:21:52.286728+00:00


Spark dihentikan. Proses Java tersisa: 3

Selesai. Lanjut ke 04_kmer_features.ipynb

Notebook 04 dan seterusnya memakai Spark. Kalau ingin menjalankannya
dengan profil standalone, kembalikan klasternya lebih dulu:
  docker compose --profile yarn down
  docker compose --profile standalone up -d

Spark juga bisa tetap berjalan di atas YARN -- sudah dibuktikan di sel
pembanding di atas -- selama seluruh transformasi memakai Spark SQL
asli tanpa UDF Python, karena NodeManager berbasis CentOS 7 tidak
membawa Python sama sekali.
